# ⛰ Core Lab 1 — Everything Is Numbers
**A Billion Small Steps · Hardware for AI · Step 1 · Book sections 1.1–1.2 · Foothold lessons 1.1–1.2**

Read the book sections first, then run this notebook from top to bottom. Runs on the **CPU** on purpose: the default runtime is fine.

**What you should see:** the best cut-off found by search; a 28 × 28 digit as 784 numbers.

| Part | What you do | Book |
|---|---|---|
| A | Let Python search for the best pass/fail cut-off | 1.1 |
| B | Meet MNIST: look inside a real image, a sentence and a tensor | 1.2 |

⏱️ About 2 hours. 📘 Textbook: D2L Ch 1, §2.1, §9.2. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **No GPU needed for this lab.** The default CPU runtime is fine — that is deliberate: you will compare against the GPU later.
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import numpy as np                 # NumPy: arrays of numbers and fast maths on them ("np" is its short name)
import matplotlib.pyplot as plt    # Matplotlib: draws charts and pictures ("plt" is its short name)
import time                        # time: we use its stopwatch, time.perf_counter()

# Print arrays neatly: wide lines, 3 decimal places, no 1e-05 style numbers
np.set_printoptions(linewidth=150, precision=3, suppress=True)
print("Ready! NumPy version:", np.__version__)   # show that everything loaded, and NumPy's version

---
## 🎬 Opening demo · the same multiply, CPU vs GPU
*(Your teacher runs this on a GPU runtime at the start of the session. On your CPU runtime it says there is no GPU, which is fine: Core Lab 5 is where you race them yourself.)*

In [ ]:
import torch                                     # PyTorch: does maths on the CPU or the GPU
n = 4096                                         # two 4,096 × 4,096 matrices: about 69 billion multiply-adds
A, B = torch.rand(n, n), torch.rand(n, n)        # random numbers, made on the CPU
t0 = time.perf_counter(); A @ B                  # start the stopwatch, multiply on the CPU
cpu_s = time.perf_counter() - t0                 # seconds the CPU took
print(f"CPU: {cpu_s:.2f} s")                     # show it
if torch.cuda.is_available():                    # is there a GPU in this session?
    Ag, Bg = A.cuda(), B.cuda()                  # copy both matrices into the GPU's memory
    Ag @ Bg; torch.cuda.synchronize()            # one warm-up multiply (the first GPU call is slow), then wait for it
    t0 = time.perf_counter()                     # start the stopwatch again
    Ag @ Bg; torch.cuda.synchronize()            # the same multiply on the GPU; wait until it has really finished
    gpu_s = time.perf_counter() - t0             # seconds the GPU took
    print(f"GPU: {gpu_s:.3f} s  →  about {cpu_s / gpu_s:.0f}× faster ({torch.cuda.get_device_name(0)})")   # the speed-up
else:                                            # a CPU runtime
    print("No GPU in this session (that's expected in Core Lab 1).")   # nothing more to show

## Part A · Rules vs Learning  *(Section 1.1)*

Last year's 12 students: study hours per day, and whether they passed (1) or failed (0).
First **you** write a rule, then Python **learns** one by trying every cut-off — exactly the search you watched in the lesson.

In [ ]:
# The data: three lists, lined up position by position (student 0 is SNEHA, 0.5 hours, failed)
names  = ["SNEHA","ARUN","ARJUN","DIVYA","RAHUL","VIJAY","NISHA","KARTHIK","MEENA","ANU","PRIYA","LAKSHMI"]
hours  = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.0, 3.5, 4.0, 4.5, 5.5, 6.0]   # study hours per day (the feature)
passed = [0,   0,   0,   0,   0,   0,   1,   1,   1,   1,   1,   1  ]   # 1 = passed, 0 = failed (the label)

def my_rule(h):   # our own rule: give it hours, it answers 1 (PASS) or 0 (FAIL)
    # ✏️ a hand-written rule: PASS (1) if hours >= 5, otherwise FAIL (0)
    return 1 if h >= 5 else 0

# "correct" counts EVERY right prediction: a correct PASS and a correct FAIL both count
correct = 0
for name, h, p in zip(names, hours, passed):     # zip walks through the three lists together
    guess = my_rule(h)                           # what our rule predicts for this student
    ok = guess == p                              # True if the prediction matches what really happened
    correct += ok                                # True counts as 1, False as 0
    print(f"{name:8s} {h:3.1f} h   rule says {'PASS' if guess else 'FAIL'}   actually {'PASS' if p else 'FAIL'}   {'✓' if ok else '✗'}")   # one line per student: hours, the rule's answer, the truth, and a tick or cross
print(f"\nMy hand-written rule gets {correct} / {len(hours)} right")   # the score (the backslash-n prints an empty line first)

Now let the machine **learn** the cut-off: try every candidate, score it against the real answers, keep the best.
This loop *is* learning, in its simplest form.

In [ ]:
candidates = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]   # every cut-off we will try
best_cut, best_score = None, -1                  # nothing found yet

for cut in candidates:   # try every candidate cut-off, one at a time
    # score = how many students the rule "PASS if hours >= cut" gets right
    score = sum(1 for h, p in zip(hours, passed) if (1 if h >= cut else 0) == p)
    print(f"cut-off {cut:>3}: {score:2d} / 12")   # show this cut-off's score
    if score > best_score:            # better than anything so far? remember it (ties keep the FIRST one)
        best_cut, best_score = cut, score   # remember it: the best so far

print(f"\nLearned rule: PASS if hours >= {best_cut}  ({best_score} / 12 correct)")   # the winner (the backslash-n prints an empty line first)

**Why not 12 / 12?** VIJAY and NISHA both studied 3.0 hours, but one failed and one passed. No single cut-off can be right for both. Real data is messy.

### ✏️ YOUR TURN (A)
1. Change `my_rule` so it gets as many right as the learned rule.
2. Add a 13th student who studied **1 hour and passed**. Re-run both cells. Does the learned cut-off change? Why or why not?

*Write your answers here (double-click to edit):*

## Part B · Everything Is Numbers  *(Section 1.2)*

Nila's teacher would love a computer that reads the marks handwritten on answer sheets. To build one, we need thousands of examples of handwritten digits, each with the right answer. That is exactly what MNIST is.

### First, what is MNIST?
**MNIST** (say "em-nist") is the most famous practice dataset in AI — the "Hello, World!" of machine learning.

- It is a collection of **70,000 small pictures of handwritten digits, 0 to 9**, written by about 500 different people (American high-school students and census-office staff).
- Each picture is **28 × 28 pixels, black-and-white (grey-scale)**: one number per pixel, from 0 (black) to 255 (white). So one picture = 28 × 28 = **784 numbers**.
- Every picture comes with its **label**: the correct answer, i.e. which digit the person meant to write.
- It is split into **60,000 training images** (the model learns from these) and **10,000 test images** (kept aside to check the model on digits it has never seen).
- The name stands for *Modified National Institute of Standards and Technology* — the US agency whose handwriting collections it was made from, in 1998.

**Why everyone starts with it:** it is free, small (about 11 MB), quick to train on even a laptop, and the task is easy to understand — *look at a picture, say which digit it is*. Yet it is a real AI problem: nobody can write exact rules for every person's handwriting (Section 1.1!). Exactly the same ideas scale up to face unlock, number-plate reading and cheque scanning.

We get MNIST from **torchvision**, PyTorch's companion library that can download famous datasets with one line. The first run downloads it into a folder called `data` (a few seconds).

In [ ]:
from torchvision import datasets       # torchvision knows how to download famous datasets like MNIST

# Download the 60,000 TRAINING images into the folder "data" (only the first time; after that it is reused)
mnist = datasets.MNIST(root="data", train=True, download=True)
print("Number of training images:", len(mnist))   # how many pictures we have

# Draw the first 10 pictures with their labels, so you can see what MNIST looks like
plt.figure(figsize=(10, 1.6))                    # a wide, short drawing area (width, height in inches)
for i in range(10):   # the first 10 pictures
    image, label = mnist[i]                      # each item is a pair: (the picture, the correct digit)
    plt.subplot(1, 10, i + 1)                    # 1 row, 10 columns: draw in box number i+1
    plt.imshow(image, cmap="gray")               # show the picture in grey-scale
    plt.title(str(label))                        # write its label above it
    plt.axis("off")                              # hide the x/y axis numbers
plt.suptitle("The first 10 MNIST training images and their labels")   # a title over all 10 pictures
plt.show()                                       # display the drawing

Now look at **one** picture the way the computer sees it.

In [ ]:
image, label = mnist[0]          # the very first training image and its label
pixels = np.array(image)         # turn the picture into a NumPy array: a 28 × 28 grid of numbers

print("This digit is a:", label)   # the digit this picture shows
# .shape = the size of each dimension; .min()/.max() = the smallest and largest number inside
print("Shape:", pixels.shape, "  smallest:", pixels.min(), "  largest:", pixels.max())

Print the raw numbers. Squint — you can see the digit drawn in the numbers themselves!

In [ ]:
np.set_printoptions(linewidth=250)   # allow very long lines, so each of the 28 rows fits on one line
print(pixels)                        # 28 rows × 28 numbers: 0 = black background, up to 255 = white ink

In [ ]:
# Turn the 784 numbers back into a picture
plt.figure(figsize=(3, 3))           # make a small 3 × 3-inch drawing area
plt.imshow(pixels, cmap="gray")      # colour each number as a shade of grey: 0 = black, 255 = white
plt.title(f"Label: {label}")         # show the correct answer (the label) above the picture
plt.axis("off")                      # hide the axis ticks and numbers — we only want the picture
plt.show()                           # display it

**Text becomes numbers too.** At the end of term Nila writes feedback on the course. Real chatbots use a dictionary of ~1 lakh tokens; here is a tiny one so you can see the idea.

In [ ]:
# A tiny "dictionary" (vocabulary): each word (token) has an ID number
vocab = {"i": 41, "love": 893, "maths": 2711, "the": 5, "class": 1204, "was": 12, "fun": 3330}

sentence = "I love maths"            # Nila's feedback on the course
tokens = sentence.lower().split()            # lower-case, then split at spaces → ['i', 'love', 'maths']
ids = [vocab.get(t, 0) for t in tokens]      # look up each token's ID; 0 = [UNK], an unknown word
print(tokens, "→", ids)   # show the words and their ID numbers

**Tensors and shapes.** A batch of 32 MNIST images stacked together is one 3-D tensor. Its *shape* says how the numbers are arranged.

In [ ]:
# Take the first 32 pictures, turn each into a 28 × 28 array, and stack them into one block
batch = np.stack([np.array(mnist[i][0]) for i in range(32)])   # mnist[i][0] = the picture of item i
print("One image:", pixels.shape, " =", pixels.size, "numbers")  # .size = total count of numbers
print("A batch  :", batch.shape, " =", batch.size, "numbers")    # (32, 28, 28): 32 images, each 28 × 28

### ✏️ YOUR TURN (B)
1. Show image number 7 instead of image 0. What digit is it?
2. A colour photo of 224 × 224 pixels has 3 numbers per pixel. How many numbers is a batch of 64 such photos? Compute it in a code cell.

*Your answers:*

---
## 🎓 Wrap-up
- Learning = trying settings and keeping the one with the smallest error, instead of writing the rule by hand.
- Everything a machine reads (a picture, a sentence, a sound) becomes a tensor of numbers first.
- **Next — Core Lab 2: How a Neuron Learns.** One neuron, a bias, the nudge test and gradient descent.